# Task 1: Data Exploration and Enrichment
**Project:** Forecasting Financial Inclusion in Ethiopia  
**Analyst:** Selam Analytics  
**Date:** 2026-07-15

## Objective
Understand the starter dataset schema and enrich it with additional data useful for forecasting Access (Account Ownership) and Usage (Digital Payment Adoption).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
COLORS = sns.color_palette('Set2', 10)

print('Libraries loaded successfully.')

## 1.1 Load the Dataset

In [ ]:
# Load all three datasets
df = pd.read_csv('../data/raw/ethiopia_fi_unified_data.csv', parse_dates=['observation_date'])
ref = pd.read_csv('../data/raw/reference_codes.csv')

print(f'Unified dataset shape: {df.shape}')
print(f'Reference codes shape: {ref.shape}')
print('\nColumns in unified dataset:')
print(df.columns.tolist())

## 1.2 Understand the Schema

In [ ]:
# Record type distribution
print('=== Record Type Distribution ===')
print(df['record_type'].value_counts())
print('\n=== Pillar Distribution ===')
print(df['pillar'].value_counts(dropna=False))
print('\n=== Source Type Distribution ===')
print(df['source_type'].value_counts(dropna=False))
print('\n=== Confidence Distribution ===')
print(df['confidence'].value_counts(dropna=False))

In [ ]:
# Visualise record type distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Record types
rt_counts = df['record_type'].value_counts()
axes[0].bar(rt_counts.index, rt_counts.values, color=[COLORS[i] for i in range(len(rt_counts))])
axes[0].set_title('Records by Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Record Type')
axes[0].set_ylabel('Count')
for i, (k, v) in enumerate(rt_counts.items()):
    axes[0].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Pillar for observations
obs = df[df['record_type'] == 'observation']
pillar_counts = obs['pillar'].value_counts()
axes[1].pie(pillar_counts.values, labels=pillar_counts.index, autopct='%1.1f%%',
            colors=[COLORS[0], COLORS[1]], startangle=90)
axes[1].set_title('Observations by Pillar', fontsize=13, fontweight='bold')

# Confidence levels
conf_counts = df[df['record_type'] == 'observation']['confidence'].value_counts()
axes[2].bar(conf_counts.index, conf_counts.values, color=[COLORS[2], COLORS[3], COLORS[4]][:len(conf_counts)])
axes[2].set_title('Observations by Confidence', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Confidence Level')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../reports/figures/task1_schema_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 1.3 Explore the Data

In [ ]:
# Temporal range of observations
obs = df[df['record_type'] == 'observation'].copy()
print('Temporal range of observations:')
print(f"  Earliest: {obs['observation_date'].min()}")
print(f"  Latest:   {obs['observation_date'].max()}")
print(f"  Total:    {len(obs)} observations\n")

# Unique indicators
print('=== Unique Indicators and Coverage ===')
indicator_summary = obs.groupby('indicator_code').agg(
    count=('value_numeric', 'count'),
    min_date=('observation_date', 'min'),
    max_date=('observation_date', 'max'),
    pillar=('pillar', 'first')
).reset_index()
print(indicator_summary.to_string(index=False))

In [ ]:
# Temporal coverage heatmap
obs_copy = obs.copy()
obs_copy['year'] = obs_copy['observation_date'].dt.year
pivot = obs_copy.pivot_table(index='indicator_code', columns='year', values='value_numeric',
                              aggfunc='count').fillna(0)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot, ax=ax, cmap='YlOrRd', linewidths=0.5, linecolor='gray',
            cbar_kws={'label': 'Data Points Available'},
            annot=True, fmt='.0f')
ax.set_title('Temporal Coverage by Indicator\n(number of data points per year)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Indicator Code')
plt.tight_layout()
plt.savefig('../reports/figures/task1_temporal_coverage.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNote: Most indicators have very sparse coverage – a key forecasting challenge.')

In [ ]:
# Explore events catalogued
events = df[df['record_type'] == 'event'].copy()
print('=== Catalogued Events ===')
event_display = events[['id', 'indicator_code', 'category', 'observation_date', 'notes']].copy()
event_display['observation_date'] = event_display['observation_date'].dt.strftime('%Y-%m')
print(event_display.to_string(index=False))

In [ ]:
# Review impact links
links = df[df['record_type'] == 'impact_link'].copy()
print('=== Impact Links Summary ===')
link_display = links[['parent_id', 'related_indicator', 'impact_direction', 'impact_magnitude', 'lag_months']].copy()
print(link_display.to_string(index=False))
print(f'\nTotal impact links: {len(links)}')
print(f'Unique events referenced: {links["parent_id"].nunique()}')
print(f'Unique indicators affected: {links["related_indicator"].nunique()}')

## 1.4 Key Design Principle: Why Events Don't Have Pillars

> A single event (e.g., Telebirr launch) can affect **both pillars simultaneously**:
> - **Access**: New mobile money accounts opened for the first time
> - **Usage**: Increased digital payment transactions
>
> Pre-assigning an event to a pillar would introduce artificial bias. `impact_link` records capture specific effects on each indicator with their own direction, magnitude, and lag.

In [ ]:
# Visualise event categories
fig, ax = plt.subplots(figsize=(8, 5))
cat_counts = events['category'].value_counts()
colors = [COLORS[i] for i in range(len(cat_counts))]
bars = ax.barh(cat_counts.index, cat_counts.values, color=colors, edgecolor='white', linewidth=1.5)
ax.set_title('Events by Category', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
for bar, val in zip(bars, cat_counts.values):
    ax.text(val + 0.05, bar.get_y() + bar.get_height()/2, str(val),
            va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/task1_event_categories.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.5 Data Quality Assessment

In [ ]:
print('=== Data Quality Assessment ===')
print('\nMissing values per column (observations only):')
obs_missing = obs.isnull().sum()
print(obs_missing[obs_missing > 0])

print('\nConfidence distribution:')
print(obs['confidence'].value_counts())
pct_high = (obs['confidence'] == 'high').mean() * 100
print(f'\n{pct_high:.1f}% of observations are high confidence (primary sources: Findex, IMF FAS, Ethio Telecom)')

print('\n=== Data Gap Summary ===')
gaps = [
    'Findex surveys only every 3 years: 2011, 2014, 2017, 2021, 2024 (5 data points for trend)',
    'Active vs registered user gap: 54M Telebirr registered ≠ 9.45% Findex mobile money ownership',
    'No regional (state/zone) level disaggregation available',
    'Historical gender data only from 2021 onwards',
    'No transaction frequency or active user threshold data',
    'Credit penetration data essentially unavailable (~0.5% penetration)',
]
for i, gap in enumerate(gaps, 1):
    print(f'  {i}. {gap}')

## 1.6 Save Enriched Dataset

In [ ]:
# Save enriched dataset to processed
df.to_csv('../data/processed/ethiopia_fi_enriched.csv', index=False)
print(f'Enriched dataset saved: {len(df)} records')
print(f'  - Observations: {(df.record_type == "observation").sum()}')
print(f'  - Events: {(df.record_type == "event").sum()}')
print(f'  - Impact Links: {(df.record_type == "impact_link").sum()}')
print(f'  - Targets: {(df.record_type == "target").sum()}')
print('\nSee data/processed/data_enrichment_log.md for full enrichment documentation.')

## Summary

| Metric | Value |
|--------|-------|
| Total records | 57 |
| Observations | 30 |
| Events | 10 |
| Impact Links | 14 |
| Targets | 3 |
| Indicators covered | 21 |
| Temporal range | 2011–2024 |
| High-confidence records | ~70% |

**Key enrichment rationale:** Added gender disaggregation, infrastructure indicators (mobile penetration, 4G, agent density), operator-side user counts (Telebirr, M-Pesa), urban/rural split, and additional events (Fayda ID, agent banking directive, rural 4G expansion, P2P/ATM crossover milestone) to build a richer causal picture of Ethiopia's financial inclusion trajectory.